# 10 · Tool comparison with projected amounts of change

In notebook 00 (section 4) every tool is told something about the amount of change in 2025 that
nobody knows when forecasting the future: our methods the observed number of each kind of change,
CLUE-S, PLUS and FLUS the observed total of each class. This notebook gives every tool what a user
would really have: **the amount projected from 2023→2024**.

**The projection:** a first-order Markov chain, the usual way to set the demand of CA-Markov, FLUS
and PLUS (Liu et al. 2017; Liang et al. 2021). The share of each class that went to each other
class in 2023→2024, times the class's pixels in 2024 (`E.projected_counts`). Our methods get the
projected number of each kind of change; CLUE-S, PLUS and FLUS the projected total of each class
(`legend_and_demand.csv`, column `pixels_2025_projected`). MOLUSCE already extrapolates
2023→2024 by itself, so its maps from notebook 00 are used as they are.

The improved method is the version with the copy of FLUS's network (notebook 09, section 4).
Everything else is as in notebook 00: the same suitability maps, the same metrics (Pontius et al.
2004, 2008; Pontius & Millones 2011), scored against ESRI's 2025 map.

In [1]:
import sys; sys.path.insert(0, "../scripts")              # shared code: landuse_experiments/scripts/
import nb_helpers as H
import esri_methods as E
import benchmark_scoring as BS
import numpy as np, pandas as pd
prof = E.grid(); g = H.guinea_mask(prof)
b, c = E.esri_cached(2024), E.esri_cached(2025)
B = BS.BENCH
LULCC, PLUS, FLUS = B / "output_lulcc" / "projected", B / "output_plus" / "projected", B / "output_flus" / "projected"
projected = E.projected_counts(2023, 2024, g)

## 1 · The projected amount against what happened

In [2]:
classes = sorted(H.ESRI_COLOURS)
observed_totals = pd.Series([((c == k) & g & E.valid_classes(b, c)).sum() for k in classes], index=classes)
start_totals = pd.Series([((b == k) & g & E.valid_classes(b)).sum() for k in classes], index=classes)
amounts = pd.DataFrame({"Class": [H.ESRI_CLASSES[k] for k in classes], "2024": start_totals.values,
                        "2025 projected": projected.sum().values, "2025 observed": observed_totals.values})
changes_projected = int(projected.values.sum() - np.trace(projected.values))
changes_observed = int(((b != c) & g & E.valid_classes(b, c)).sum())
print(f"Changes: projected {changes_projected:,}, observed {changes_observed:,}")
H.show(amounts)

Changes: projected 428,139, observed 507,539


Class,2024,2025 projected,2025 observed
Water,22447,21048,24555
Trees,2109123,2028750,2351004
Flooded Vegetation,29353,29975,31919
Crops,52164,58178,51545
Built Area,44075,40427,46066
Bare Ground,904,1048,648
Rangeland,1752610,1831250,1504939


**2025 broke the 2023→2024 trend.** In 2023→2024 trees shrank and rangeland grew; in 2025 trees
grew back to 2.35 million pixels and rangeland fell to 1.50 million. The projection carries the old
trend forward, so it gets the direction of the two largest classes wrong, and it forecasts 428,139
changes where 507,539 happened. Every tool gets the same handicap: this is what a real forecast of
the next year faces.

## 2 · The comparison

In [3]:
TOOLS = ["No change", "MOLUSCE ANN", "MOLUSCE LR", "CLUE-S", "PLUS", "FLUS", "Deployed method", "Improved method"]
# Improved method: contagion features, a logistic regression and the copy of FLUS's network, averaged
# with equal weight (notebook 09, section 4; seed 42). Its version before FLUS's network: tag contagion_logistic.
ours = {"Deployed method": E.atlas_from(E.RUNS / "R0c_guinea_dated" / "test" / "surfaces", "expander"),
        "Improved method": E.atlas_from(E.RUNS / "improved_flus_copy" / "test" / "surfaces", "avg")}
TAG = {"Deployed method": "deployed_projected", "Improved method": "improved_flus_copy_projected"}

def projected_run(name):
    path = E.RUNS / TAG[name] / "test" / "allocation" / "Expander-only" / "predicted_land_cover.tif"
    if not path.exists():
        E.allocate_and_score(b, c, {"Expander-only": ours[name]}, g, path.parents[1], prof, counts=projected)
    return H.read_tif(path)[0]

maps = {"No change": b.copy(),
        "MOLUSCE ANN": BS.to_esri(H.read_tif(B / "output_molusce_2" / "molusce_ann_2025_2.tif")[0]),
        "MOLUSCE LR": BS.to_esri(H.read_tif(B / "output_molusce_2" / "molusce_lr_2025_2.tif")[0]),
        "CLUE-S": BS.to_esri(H.read_tif(LULCC / "lulcc_clues_2025.tif")[0]),        # benchmark_lulcc.R, DEMAND "projected"
        **{n: projected_run(n) for n in ours}}
for name, path in (("PLUS", next(PLUS.glob("plus_2025_projected*.tif"), PLUS / "-")),   # CARS adds "Simulation_1"
                   ("FLUS", FLUS / "flus_2025_projected.tif")):
    if path.exists():                                                          # run in their programs (BENCHMARK_GUIDE.md)
        maps[name] = BS.to_esri(H.read_tif(path)[0])

rows = {n: BS.tool_row(b, c, s, g) for n, s in maps.items()}
SHOW = {"Changes forecast": "{:,.0f}", "Figure of Merit %": "{:.1f}", "Producer's accuracy %": "{:.1f}",
        "User's accuracy %": "{:.1f}", "× random": "{:.1f}", "Quantity disagreement %": "{:.2f}",
        "Allocation disagreement %": "{:.2f}", "Figure of Merit % at 1 km": "{:.1f}", "Figure of Merit % at 5 km": "{:.1f}"}
fmt = lambda v, f: f.format(v) if v is not None and np.isfinite(v) else "–"
projected_table = pd.DataFrame({n: {k: fmt(rows[n][k], f) if n in rows else "–" for k, f in SHOW.items()} for n in TOOLS})
print(f"2024→2025, Guinea, 250 m, amounts projected from 2023→2024. Pixels that really changed: {changes_observed:,}.")
H.show(projected_table.reset_index(names="Metric"))
S = H.OUTPUTS / "summary"
projected_table.to_csv(S / "10_projected.csv")

2024→2025, Guinea, 250 m, amounts projected from 2023→2024. Pixels that really changed: 507,539.


**The same table without the deployed method:**

In [4]:
H.show(projected_table.drop(columns="Deployed method").reset_index(names="Metric"))

Metric,No change,MOLUSCE ANN,MOLUSCE LR,CLUE-S,PLUS,FLUS,Improved method
Changes forecast,0,"3,043","65,794","85,693","89,089","351,395","422,773"
Figure of Merit %,0.0,0.1,3.6,1.6,1.1,10.1,13.1
Producer's accuracy %,0.0,0.1,4.0,1.8,1.3,15.5,21.1
User's accuracy %,–,11.4,30.5,10.6,7.2,22.3,25.4
× random,–,1.5,2.4,2.4,1.6,2.2,2.5
Quantity disagreement %,6.20,6.24,6.20,8.31,8.31,8.31,8.31
Allocation disagreement %,6.46,6.47,7.04,5.97,6.22,9.11,9.44
Figure of Merit % at 1 km,0.0,0.2,5.6,2.8,3.7,17.6,17.1
Figure of Merit % at 5 km,0.0,0.4,10.0,6.1,9.3,29.5,26.1


**Reading the table:**

* **With projected amounts, the improved method is still the most accurate: 13.1%**, against 12.2%
  for the deployed method, 10.1% for FLUS, 3.6% for MOLUSCE LR, 1.6% for CLUE-S and 1.1% for PLUS.
* **Every forecast loses accuracy when the amount is not known.** The improved method drops from
  20.4% (notebook 00) to 13.1%. Quantity disagreement rises from 0.08% to 8.31%: the projection gets
  the amount wrong, which no placement can repair.
* **CLUE-S and PLUS suffer most** (14.8% → 1.6% and 12.6% → 1.1%): given class totals that move in
  the wrong direction for trees and rangeland, they forecast few changes (85,693 and 89,089). FLUS
  keeps swapping many pixels (351,395 changes) and holds up better (18.1% → 10.1%).
* **At 1 km and 5 km FLUS is ahead** (17.6% and 29.5%, against 17.1% and 26.1% for the improved
  method). With observed amounts FLUS was level at 1 km and behind at 5 km (notebook 00): forecasting
  many changes helps at coarse cells when the amount is uncertain. The deployed method is level with
  the improved method at these scales (17.0% and 26.4%).
* This is the score to expect from an operational forecast of the next year, and the fair basis for
  comparing tools: no tool is told anything about 2025.

### References

* Liang, X., Guan, Q., Clarke, K.C., et al. (2021). Understanding the drivers of sustainable land
  expansion using a patch-generating land use simulation (PLUS) model: a case study in Wuhan, China.
  *Computers, Environment and Urban Systems* 85, 101569.
* Liu, X., Liang, X., Li, X., et al. (2017). A future land use simulation model (FLUS) for simulating
  multiple land use scenarios by coupling human and natural effects. *Landscape and Urban Planning*
  168, 94–116.
* Pontius, R.G. Jr., Huffaker, D., Denman, K. (2004). Useful techniques of validation for spatially
  explicit land-change models. *Ecological Modelling* 179(4), 445–461.
* Pontius, R.G. Jr., Boersma, W., Castella, J.-C., et al. (2008). Comparing the input, output, and
  validation maps for several models of land change. *Annals of Regional Science* 42(1), 11–37.
* Pontius, R.G. Jr., Millones, M. (2011). Death to Kappa: birth of quantity disagreement and
  allocation disagreement for accuracy assessment. *International Journal of Remote Sensing* 32(15),
  4407–4429.